In [1]:
from collections import Counter
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\dataset1_729×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape


(729, 87)

In [2]:
def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage1\benchmark1.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)

for j in range(30, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]
    for i in range(3, 7):

                model = AgglomerativeClustering(n_clusters= i, metric = 'euclidean', linkage = 'ward')
                model = model.fit(pca_data)
                labels = model.labels_
                if len(np.unique(labels)) > 1:

                    print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")
                else:
                    pass

                data[f'cluster_{i}'] = labels
                print(f"Number of clusters k={i}:",f"\nPCA={j}:")
                dict = {}

                label_1 = []
                for num  in  reported_structures:
                    label = labels[num-1]
                    label_1.append(int(label))
                print(label_1)

                for cluster in range(i):

                    cluster_samples = data[data[f'cluster_{i}'] == cluster]

                    total_in_cluster = len(cluster_samples)

                    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])
                    dict[cluster] = (reported_in_cluster, total_in_cluster)
                for value in dict.values():
                    if value[0] == 0:
                        for key, value in dict.items():

                            formatted_value = f"{value[0]}/{value[1]}"

                            print(f"Cluster {key}: {formatted_value}")


[22, 23, 24, 25, 260, 300, 384, 387, 400, 404, 423, 435, 489, 641, 703]

Silhouette score: 0.53 
Calinski-Harabasz score: 842.53
Number of clusters k=3: 
PCA=0.3:
[0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0]
Cluster 0: 9/427
Cluster 1: 6/247
Cluster 2: 0/55

Silhouette score: 0.45 
Calinski-Harabasz score: 807.57
Number of clusters k=4: 
PCA=0.3:
[1, 1, 1, 0, 3, 0, 0, 0, 0, 1, 3, 0, 1, 1, 1]
Cluster 0: 6/247
Cluster 1: 7/334
Cluster 2: 0/55
Cluster 3: 2/93

Silhouette score: 0.46 
Calinski-Harabasz score: 884.36
Number of clusters k=5: 
PCA=0.3:
[0, 0, 0, 1, 3, 1, 1, 1, 4, 0, 3, 1, 0, 0, 0]
Cluster 0: 7/334
Cluster 1: 5/182
Cluster 2: 0/55
Cluster 3: 2/93
Cluster 4: 1/65

Silhouette score: 0.42 
Calinski-Harabasz score: 1026.59
Number of clusters k=6: 
PCA=0.3:
[1, 1, 1, 0, 3, 0, 0, 0, 4, 2, 3, 0, 1, 1, 1]
Cluster 0: 5/182
Cluster 1: 6/183
Cluster 2: 1/151
Cluster 3: 2/93
Cluster 4: 1/65
Cluster 5: 0/55

Silhouette score: 0.40 
Calinski-Harabasz score: 420.60
Number of clusters k=3: 


In [3]:
pca = PCA(n_components= 0.6, random_state=42)
new_data = pca.fit_transform(X)
print(new_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)

i = 6
model = AgglomerativeClustering(n_clusters= i, metric = 'euclidean', linkage = 'ward')
model = model.fit(new_data)
labels = model.labels_

df = pd.DataFrame(labels)
df.to_excel("ward_result_1.xlsx", index=False)

data[f'cluster_{i}'] = labels

print(f"\nNumber of clusters k={i}:")
print(f"\nSilhouette score: {silhouette_score(new_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(new_data, labels):.2f}")
label_1 = []
for num  in  reported_structures:
        label = labels[num-1]
        label_1.append(int(label))
print(label_1)

for cluster in range(i):

    cluster_samples = data[data[f'cluster_{i}'] == cluster]

    total_in_cluster = len(cluster_samples)
    print(total_in_cluster)

    reported_in_cluster = len([num for num in cluster_samples['number'] if num in reported_structures])

    print(f"Cluster  {cluster}: {reported_in_cluster}/{total_in_cluster} "
                f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


(729, 6)

Number of clusters k=6:

Silhouette score: 0.23 
Calinski-Harabasz score: 190.70
[1, 4, 4, 0, 1, 0, 0, 0, 0, 4, 1, 0, 3, 3, 1]
172
Cluster  0: 6/172 (Reported ratio: 3.49%)
102
Cluster  1: 4/102 (Reported ratio: 3.92%)
64
Cluster  2: 0/64 (Reported ratio: 0.00%)
175
Cluster  3: 2/175 (Reported ratio: 1.14%)
162
Cluster  4: 3/162 (Reported ratio: 1.85%)
54
Cluster  5: 0/54 (Reported ratio: 0.00%)


In [4]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook

wb = Workbook()
ws = wb.active

for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)

red_bold_font = Font(color="FF0000", bold=True)

for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]

    values = [0 if v is None else v for v in values]

    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]

    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font

output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")


The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
